traitement données base gravity répliquée de WR avec base gravity du CEPII pour données économiques (gdp) 

In [31]:
import pandas as pd
import numpy as np

# --- 1. CONFIGURATION ---
path_r_output = "/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/gravity_data_WR_replique_sans_GDP.csv"
path_gravity  = "/Users/romain/Desktop/Projets DS/ProjetStat/data/GravityCEPII/Gravity_V202211.csv"
years_needed  = [1985, 1990, 1995, 2000, 2005, 2010, 2015]

# --- 2. CRÉATION DU DICTIONNAIRE MONADIQUE (PAYS-ANNÉE) ---
print("Création du dictionnaire PIB unique (Monadique)...")

cols_eco = ['year', 'iso3_o', 'gdp_o', 'gdpcap_o'] 

# On charge sans chunking pour être sûr (optimisé RAM par usecols)
df_gdp = pd.read_csv(path_gravity, usecols=cols_eco, 
                     keep_default_na=False, na_values=[''])

# Filtre et Nettoyage
df_gdp = df_gdp[df_gdp['year'].isin(years_needed)].copy()
df_gdp['iso3_o'] = df_gdp['iso3_o'].astype(str).str.strip()
df_gdp['year'] = df_gdp['year'].astype(int)

# Conversion numérique sécurisée
for col in ['gdp_o', 'gdpcap_o']:
    df_gdp[col] = pd.to_numeric(df_gdp[col], errors='coerce')

# ASTUCE CRUCIALE : On trie par PIB décroissant avant de dédoublonner
# Comme ça, si la France apparait 100 fois dont 1 fois avec NaN, 
# on garde une ligne avec le PIB rempli (les NaN vont à la fin).
df_gdp = df_gdp.sort_values(by='gdp_o', ascending=False, na_position='last')
df_gdp = df_gdp.drop_duplicates(subset=['year', 'iso3_o'])

print(f"✅ Dictionnaire Monadique prêt : {len(df_gdp)} lignes (Pays-Années uniques).")

# --- 3. CHARGEMENT ET PRÉPARATION MAIN ---
df_main = pd.read_csv(path_r_output)
df_main['orig'] = df_main['orig'].astype(str).str.strip()
df_main['dest'] = df_main['dest'].astype(str).str.strip()
df_main['year0'] = df_main['year0'].astype(int)

# --- 4. FUSION INTELLIGENTE (2 FOIS) ---

# A. Fusion pour l'ORIGINE (gdp_o)
print("Fusion PIB Origine...")
df_final = pd.merge(
    df_main,
    df_gdp,
    left_on=['year0', 'orig'],
    right_on=['year', 'iso3_o'],
    how='left'
)
df_final = df_final.drop(columns=['year', 'iso3_o']) 

# B. Fusion pour la DESTINATION (gdp_d)
print("Fusion PIB Destination...")
df_final = pd.merge(
    df_final,
    df_gdp,
    left_on=['year0', 'dest'],
    right_on=['year', 'iso3_o'],
    how='left',
    suffixes=('', '_d_raw') 
)
# Renommage propre
df_final = df_final.rename(columns={'gdp_o_d_raw': 'gdp_d', 'gdpcap_o_d_raw': 'gdpcap_d'})
df_final = df_final.drop(columns=['year', 'iso3_o'])

# --- 5. GESTION DES LAGS (t-5) ---
print("Fusion des Lags...")
df_gdp_lag = df_gdp.copy()
df_gdp_lag['year_target'] = df_gdp_lag['year'] + 5
df_gdp_lag = df_gdp_lag.rename(columns={'gdp_o': 'gdp_o_lag', 'gdpcap_o': 'gdpcap_o_lag'})

# Lag Origine
df_final = pd.merge(
    df_final,
    df_gdp_lag[['year_target', 'iso3_o', 'gdp_o_lag', 'gdpcap_o_lag']],
    left_on=['year0', 'orig'],
    right_on=['year_target', 'iso3_o'],
    how='left'
).drop(columns=['year_target', 'iso3_o'])

# Lag Destination
df_final = pd.merge(
    df_final,
    df_gdp_lag[['year_target', 'iso3_o', 'gdp_o_lag', 'gdpcap_o_lag']],
    left_on=['year0', 'dest'],
    right_on=['year_target', 'iso3_o'],
    how='left',
    suffixes=('', '_d_lag')
).drop(columns=['year_target', 'iso3_o'])

df_final = df_final.rename(columns={'gdp_o_lag_d_lag': 'gdp_d_lag', 'gdpcap_o_lag_d_lag': 'gdpcap_d_lag'})

# --- 6. LOGS & EXPORT ---
vars_to_log = ['gdp_o', 'gdp_d', 'gdpcap_o', 'gdpcap_d', 'gdp_o_lag', 'gdp_d_lag', 'gdpcap_o_lag', 'gdpcap_d_lag']
for col in vars_to_log:
    df_final[f'log_{col}'] = np.log(df_final[col].mask(df_final[col] <= 0))

# Nettoyage final
df_final.rename(columns={'year0': 'year', 'orig': 'iso_o', 'dest': 'iso_d'}, inplace=True)
cols_to_drop = ['iso3.x', 'iso3.y'] 
df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns], inplace=True)

print("-" * 30)
print(f"Dimensions finales : {df_final.shape}")
print(f"PIB manquants (courant) : {df_final['gdp_o'].isna().sum()} / {len(df_final)}")

df_final=df_final.rename(columns={'iso_o': 'orig', 'iso_d': 'dest', 'cod_d':'iso3_d','cod_o':'iso3_o'})

output_path = "/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/DF_GRAVITY_NaN_a_remplir.csv"
df_final.to_csv(output_path, index=False)
print(f"✅ Fichier sauvegardé : {output_path}")

Création du dictionnaire PIB unique (Monadique)...
✅ Dictionnaire Monadique prêt : 1701 lignes (Pays-Années uniques).
Fusion PIB Origine...
Fusion PIB Destination...
Fusion des Lags...
------------------------------
Dimensions finales : (229350, 44)
PIB manquants (courant) : 23185 / 229350
✅ Fichier sauvegardé : /Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/DF_GRAVITY_NaN_a_remplir.csv


In [32]:
import pandas as pd

# Chargement de ton fichier tout frais
df = pd.read_csv("/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/DF_GRAVITY_NaN_a_remplir.csv")

# 1. On isole les coupables
missing = df[df['gdp_o'].isna()]

# 2. Qui sont-ils ? (Triés par nombre de lignes perdues)
print("--- TOP PAYS SANS PIB (Candidats au Mapping) ---")
# On compte combien de fois ils apparaissent comme ORIGINE manquante
suspects = missing['orig'].value_counts().head(20)
print(suspects)

# 3. Vérification temporelle (Est-ce que c'est juste une année ?)
print("\n--- ANNÉES CONCERNÉES ---")
print(missing.groupby('orig')['year'].unique().head(10))

--- TOP PAYS SANS PIB (Candidats au Mapping) ---
orig
VIR    1170
REU    1170
GUF    1170
GLP    1170
MYT    1170
ESH    1170
PSE    1170
MTQ    1170
PRK     977
SOM     779
TLS     777
PYF     591
NCL     591
AFG     579
GUM     579
CUW     396
SYR     396
SDN     393
ABW     391
ERI     391
Name: count, dtype: int64

--- ANNÉES CONCERNÉES ---
orig
ABW          [1990, 2015]
AFG    [1990, 1995, 2000]
ARM                [1990]
AZE                [1990]
BIH                [1990]
BLR                [1990]
COD                [2015]
CUW          [2010, 2015]
CZE                [1990]
ERI          [1990, 2015]
Name: year, dtype: object


In [33]:
# Codes synonymes potentiels (Ancien -> Nouveau ou Alternatif)
synonyms = {
    'TLS': 'TMP',  # Timor-Leste
    'PSE': 'WBG',  # Palestine -> West Bank and Gaza
    'COD': 'ZAR',  # RDC (Zaire)
    'ROU': 'ROM',  # Roumanie
    'XKX': 'KOS'   # Kosovo
}

print("--- RECHERCHE DE SYNONYMES DANS GRAVITY ---")
# On regarde si le code "Synonyme" existe dans le dictionnaire df_gdp qu'on a créé
# (Note: df_gdp est ton dictionnaire monadique en mémoire)

available_codes = df_gdp['iso3_o'].unique()

for current, old in synonyms.items():
    if old in available_codes:
        print(f"✅ ESPOIR : '{old}' existe dans Gravity (pour remplacer '{current}')")
    else:
        print(f"❌ '{old}' n'existe pas non plus.")

--- RECHERCHE DE SYNONYMES DANS GRAVITY ---
❌ 'TMP' n'existe pas non plus.
❌ 'WBG' n'existe pas non plus.
❌ 'ZAR' n'existe pas non plus.
❌ 'ROM' n'existe pas non plus.
❌ 'KOS' n'existe pas non plus.


In [34]:


# 1. CHARGEMENT (Ta dernière version Monadique)
path_monadic = "/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/DF_GRAVITY_NaN_a_remplir.csv"
df_final = pd.read_csv(path_monadic)

# 2. PAS DE MAPPING (Puisque le diagnostic synonyme a échoué)
# On passe directement au nettoyage.

# 3. SUPPRESSION DES DONNÉES MANQUANTES
# On ne garde que les lignes où l'on a TOUS les PIB (Origine et Destination)
df_clean = df_final.dropna(subset=['gdp_o', 'gdp_d'])

# 4. STATISTIQUES FINALES
perte = len(df_final) - len(df_clean)
pourcent = perte / len(df_final)

print("-" * 30)
print(f"Base Initiale : {len(df_final):,} lignes")
print(f"Base Nettoyée : {len(df_clean):,} lignes")
print(f"Lignes supprimées (Manque PIB) : {perte:,} ({pourcent:.1%})")
print("-" * 30)


# 5. EXPORT FINAL (READY FOR RANDOM FOREST)
path_clean = "/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/DF_GRAVITY_sans_NaN.csv"
df_clean.to_csv(path_clean, index=False)
print(f"🚀 Fichier FINAL prêt pour le modèle : {path_clean}")

------------------------------
Base Initiale : 229,350 lignes
Base Nettoyée : 185,578 lignes
Lignes supprimées (Manque PIB) : 43,772 (19.1%)
------------------------------
🚀 Fichier FINAL prêt pour le modèle : /Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/DF_GRAVITY_sans_NaN.csv


10% de trous monoadiques (pays sans PIB)

$10\% + 10\% - (overlap) \approx 19\%$ de couples à trous




PAYS SUPPRIMES DE L'ANALYSE: 

Les DOM-TOM : Réunion, Martinique, Guyane, Guadeloupe... (Le CEPII n'a pas leur PIB, ils utilisent celui de la France ou rien).

Les États en crise : Corée du Nord, Somalie, Afghanistan.

Les Micro-États : Nauru, Tuvalu, Saint-Marin...

19% de lignes manquantes: intégrer les données de l'INSEE sur la réunion, etc. Chercher PIB microétats. Atteindre le maximum théorique. Autres données manquantes que le PIB pour ces zones?  

possible chunks si trop long à mettre en cellule de code: 

import pandas as pd
import polars as pl

#df_gravity= pd.read_csv('/Users/romain/Desktop/Projets DS/ProjetStat/data/GravityCEPII/Gravity_V202211.csv',)
#df_gravity=df_gravity[df_gravity['year']==2010 | df_gravity['year']==2015 | df_gravity['year']==2005 | df_gravity['year']==2000 | df_gravity['year']==1995 | df_gravity['year']==1990]


#CHUNKS FAIT PAR GEMINI

PATH_GRAVITY = '/Users/romain/Desktop/Projets DS/ProjetStat/data/GravityCEPII/Gravity_V202211.csv'
ANNEES_CIBLES = [1990, 1995, 2000, 2005, 2010, 2015]

# Liste pour stocker les petits morceaux filtrés
chunks_list = []



# chunksize=100000 signifie qu'on lit 100 000 lignes à la fois
# low_memory=False permet d'éviter les warnings sur les types mixtes (ex: 6.5 vs 6)
with pd.read_csv(PATH_GRAVITY, chunksize=100000, low_memory=False) as reader:
    for i, chunk in enumerate(reader):
        # 1. On ne garde que les années cibles dans ce morceau
        chunk_filtered = chunk[chunk['year'].isin(ANNEES_CIBLES)]
        
        # 2. On ajoute le résultat à notre liste (si le morceau n'est pas vide)
        if not chunk_filtered.empty:
            chunks_list.append(chunk_filtered)
            
        # Petit indicateur de progression (optionnel)
        if i % 10 == 0:
            print(f"   ... Traitement du bloc {i}")


if len(chunks_list) > 0:
    df_gravity = pd.concat(chunks_list, ignore_index=True)
    print(f"Terminé ! Taille finale : {df_gravity.shape}")
    print("Années présentes :", sorted(df_gravity['year'].unique()))
else:
    print("Attention : Aucune donnée trouvée pour ces années.")

# Affichage des premières lignes
print(df_gravity.head())